# Pràctica 4: Embeddings distribucionals i contextuals

## Introducció

L'objectiu principal d'aquesta pràctica és entrenar i avaluar models d'**embeddings distribucionals i contextuals** per a tasques de similitud semàntica en espanyol.

Els **embeddings distribucionals** (com Word2Vec o GloVe) representen paraules com a vectors densos basant-se en la hipòtesi distribucional: paraules que apareixen en contextos similars tendeixen a tenir significats similars. Aquests vectors es calculen a partir d'un corpus gran i són estàtics, és a dir, cada paraula té sempre el mateix vector independentment del context.

Els **embeddings contextuals** (com BERT o RoBERTa) superen aquesta limitació generant representacions dinàmiques que canvien segons el context de la frase. Això permet capturar fenòmens com la polisèmia.

## Datasets

- **Multi-SimLex (Spanish):** conjunt de parells de paraules amb puntuacions de similitud semàntica anotades per humans. S'utilitza per a l'**avaluació intrínseca**, mesurant directament la qualitat dels embeddings mitjançant la correlació de Spearman entre les similituds predites i les humanes.

- **Spanish STS:** conjunt de parells de frases amb puntuacions de similitud semàntica. S'utilitza per a l'**avaluació extrínseca** (STS), mesurant la correlació de Pearson entre les similituds predites i les de referència.

- **Corpus Wikipedia en espanyol (`raw.es.tgz`):** corpus de text en cru utilitzat per entrenar els embeddings distribucionals.

## Mètriques d'avaluació

| Dataset | Mètrica |
|---|---|
| Multi-SimLex | Correlació de Spearman (ρ) |
| Spanish STS | Correlació de Pearson (r) |




## 0. Setup

In [1]:
#%pip install requests "datasets<3.0"

In [3]:
!pip install pandas scipy datasets tqdm

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
     ---------------------------------------- 0.0/41.7 kB ? eta -:--:--
     ---------------------------------------- 41.7/41.7 kB 2.1 MB/s eta 0:00:00
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------- 82.2/82.2 kB 2.3 MB/s eta 0:00:00
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from pathlib import Path
from datasets import load_dataset
from tqdm.notebook import tqdm

c:\Users\crost\Documents\GitHub\P4plh\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Datasets


Aquesta pràctica utilitza tres datasets:

| Dataset | Descripció | Ús |
|---|---|---|
| **Multi-SimLex** | Parells de paraules amb puntuacions de similitud semàntica | Avaluació intrínseca |
| **Spanish STS** | Parells de frases amb puntuacions de similitud semàntica | Avaluació extrínseca (STS) |
| **raw_es** | Corpus de la Wikipedia en espanyol | Entrenament d'embeddings |

A partir de `raw_es`, entrenarem i compararem models de **Word2Vec** i **fastText**.

In [4]:
URL_TRANSLATIONS = "https://multisimlex.com/data/translation.csv"
URL_SCORES       = "https://multisimlex.com/data/scores.csv"

translations = pd.read_csv(URL_TRANSLATIONS)
scores       = pd.read_csv(URL_SCORES)

# Paraules espanyoles
spa_words  = translations[['PAIR_ID', 'SPA_W1', 'SPA_W2']]

# Puntuacions espanyoles (una columna per anotador)
spa_score_cols = [c for c in scores.columns if c.startswith('SPA_')]
spa_scores = scores[['PAIR_ID'] + spa_score_cols].copy()
spa_scores['SPA_AVG'] = spa_scores[spa_score_cols].mean(axis=1)  # gold standard

# Unim en un sol DataFrame
multi_simlex = spa_words.merge(spa_scores[['PAIR_ID', 'SPA_AVG']], on='PAIR_ID')

print(f"Multi-SimLex (ES) — {len(multi_simlex)} parells de paraules")
multi_simlex.head()

URLError: <urlopen error [WinError 10060] Se produjo un error durante el intento de conexión ya que la parte conectada no respondió adecuadamente tras un periodo de tiempo, o bien se produjo un error en la conexión establecida ya que el host conectado no ha podido responder>

In [12]:
# ============================================================
# DATASET 2 — Spanish STS (PlanTL-GOB-ES/sts-es)
# Avaluació extrínseca · Mètrica: correlació de Pearson
# ============================================================

ds_sts = load_dataset("PlanTL-GOB-ES/sts-es")

In [13]:
# ============================================================
# DATASET 3 — Corpus Wikipedia en espanyol
# Carpeta raw_es/ ja descomprimida, fitxers sense extensió
# ============================================================

corpus_dir = Path("raw_es")
corpus_files = sorted(corpus_dir.iterdir())

print(f"Fitxers trobats: {len(corpus_files)}")
print(f"Exemples: {[f.name for f in corpus_files[:5]]}")

def corpus_iterator(files):
    for filepath in files:
        with open(filepath, encoding="latin-1") as f:
            yield from f

# Vista prèvia
for i, line in enumerate(corpus_iterator(corpus_files)):
    print(line, end="")
    if i >= 4: break

Fitxers trobats: 57
Exemples: ['spanishText_10000_15000', 'spanishText_110000_115000', 'spanishText_120000_125000', 'spanishText_15000_20000', 'spanishText_180000_185000']
<doc id="20540" title="658" nonfiltered="1" processed="1" dbindex="10000">

 Acontecimientos .


